# Preprocesamiento Completo - Breast Cancer Wisconsin (Original)

Objetivo de clasificacion:
- Benigno (2)
- Maligno (4)

Flujo aplicado con estilo de cuadernillo:
1. Carga y descripcion del dataset
2. Limpieza inicial y split 80/20
3. Manejo de valores nulos
4. Funciones auxiliares
5. Normalizacion y codificacion de la clase
6. Balanceo de clases
7. Conversor a tensores (PyTorch)

In [1]:
# ==========================================
# 1. IMPORTACION DE LIBRERIAS
# ==========================================
import os
import numpy as np
import pandas as pd
import torch
from matplotlib import pyplot
from IPython.display import display

%matplotlib inline

## Paso 1: Carga del dataset
Se carga `breast-cancer-wisconsin.data` (UCI Original). La columna de clase usa 2=benigno y 4=maligno.

In [2]:
ruta_archivo = os.path.join(
    'Datasets Primer Parcial',
    '8-Breast Cancer Wisconsin (Original)',
    'breast+cancer+wisconsin+original',
    'breast-cancer-wisconsin.data'
)

columnas = [
    'Sample_code_number',
    'Clump_Thickness',
    'Uniformity_of_Cell_Size',
    'Uniformity_of_Cell_Shape',
    'Marginal_Adhesion',
    'Single_Epithelial_Cell_Size',
    'Bare_Nuclei',
    'Bland_Chromatin',
    'Normal_Nucleoli',
    'Mitoses',
    'Class'
]

df = pd.read_csv(ruta_archivo, header=None, names=columnas, na_values='?')

print('Dimensiones originales:', df.shape)
print('Primeras filas:')
display(df.head())
print('Nulos por columna:')
print(df.isnull().sum())

Dimensiones originales: (699, 11)
Primeras filas:


,Sample_code_number,Clump_Thickness,Uniformity_of_Cell_Size,Uniformity_of_Cell_Shape,Marginal_Adhesion,Single_Epithelial_Cell_Size,Bare_Nuclei,Bland_Chromatin,Normal_Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,2
1,1002945,5,4,4,5,7,10.0,3,2,1,2
2,1015425,3,1,1,1,2,2.0,3,1,1,2
3,1016277,6,8,8,1,3,4.0,3,7,1,2
4,1017023,4,1,1,3,2,1.0,3,1,1,2


Nulos por columna:
Sample_code_number              0
Clump_Thickness                 0
Uniformity_of_Cell_Size         0
Uniformity_of_Cell_Shape        0
Marginal_Adhesion               0
Single_Epithelial_Cell_Size     0
Bare_Nuclei                    16
Bland_Chromatin                 0
Normal_Nucleoli                 0
Mitoses                         0
Class                           0
dtype: int64


## Paso 2: Limpieza inicial y split 80/20
Se elimina el identificador `Sample_code_number`, se define la clase y se divide con indices aleatorios tipo cuadernillo.

In [3]:
if 'Sample_code_number' in df.columns:
    df = df.drop(columns=['Sample_code_number'])

target_col = 'Class'
feature_cols = [c for c in df.columns if c != target_col]

torch.manual_seed(42)

n_total = df.shape[0]
n_test = int(0.2 * n_total)
n_train = n_total - n_test

indices = torch.randperm(n_total).tolist()
train_indices = indices[:n_train]
test_indices = indices[n_train:]

train_df = df.iloc[train_indices].copy()
test_df = df.iloc[test_indices].copy()

print(f'Train: {len(train_df)}/{n_total} ({len(train_df)/n_total*100:.1f}%)')
print(f'Test : {len(test_df)}/{n_total} ({len(test_df)/n_total*100:.1f}%)')
print('Distribucion en train (Class):')
print(train_df[target_col].value_counts().sort_index())
print('Distribucion en test (Class):')
print(test_df[target_col].value_counts().sort_index())

Train: 560/699 (80.1%)
Test : 139/699 (19.9%)
Distribucion en train (Class):
Class
2    367
4    193
Name: count, dtype: int64
Distribucion en test (Class):
Class
2    91
4    48
Name: count, dtype: int64


## Paso 3: Manejo de valores nulos
Todos los atributos son numericos en escala 1-10. Se imputa mediana usando solo train y se aplica a test.

In [4]:
X_train_raw = train_df[feature_cols].copy()
X_test_raw = test_df[feature_cols].copy()
y_train_raw = train_df[target_col].copy()
y_test_raw = test_df[target_col].copy()

# Asegurar tipo numerico
for col in feature_cols:
    X_train_raw[col] = pd.to_numeric(X_train_raw[col], errors='coerce')
    X_test_raw[col] = pd.to_numeric(X_test_raw[col], errors='coerce')

print('Nulos antes (train):', X_train_raw.isnull().sum().sum())
print('Nulos antes (test) :', X_test_raw.isnull().sum().sum())

for col in feature_cols:
    med = X_train_raw[col].median()
    X_train_raw[col] = X_train_raw[col].fillna(med)
    X_test_raw[col] = X_test_raw[col].fillna(med)

print('Nulos despues (train):', X_train_raw.isnull().sum().sum())
print('Nulos despues (test) :', X_test_raw.isnull().sum().sum())

Nulos antes (train): 14
Nulos antes (test) : 2
Nulos despues (train): 0
Nulos despues (test) : 0


## Paso 4: Funciones auxiliares
Se define `featureNormalize(X)` con el formato de los cuadernillos.

In [5]:
def featureNormalize(X):
    X_norm = X.copy()
    mu = np.zeros(X.shape[1])
    sigma = np.zeros(X.shape[1])

    mu = np.mean(X, axis = 0)
    sigma = np.std(X, axis = 0)
    X_norm = (X - mu) / sigma

    return X_norm, mu, sigma

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

## Paso 5: Normalizacion y codificacion de la clase
Se normalizan los atributos con estadisticos de train y la clase se mapea a binario: 2->0 (benigno), 4->1 (maligno).

In [6]:
X_train_num = X_train_raw.to_numpy(dtype=float)
X_test_num = X_test_raw.to_numpy(dtype=float)

X_train_norm, mu_norm, sigma_norm = featureNormalize(X_train_num)
X_test_norm = (X_test_num - mu_norm) / sigma_norm

X_train_final_df = pd.DataFrame(X_train_norm, columns=feature_cols, index=X_train_raw.index)
X_test_final_df = pd.DataFrame(X_test_norm, columns=feature_cols, index=X_test_raw.index)

class_map = {2: 0, 4: 1}
y_train = y_train_raw.map(class_map).to_numpy(dtype=np.int64)
y_test = y_test_raw.map(class_map).to_numpy(dtype=np.int64)

print('X_train final:', X_train_final_df.shape)
print('X_test final :', X_test_final_df.shape)
print('Clases binarias (0=benigno, 1=maligno) en train:')
print(pd.Series(y_train).value_counts().sort_index())
display(X_train_final_df.head())

X_train final: (560, 9)
X_test final : (139, 9)
Clases binarias (0=benigno, 1=maligno) en train:
0    367
1    193
Name: count, dtype: int64


,Clump_Thickness,Uniformity_of_Cell_Size,Uniformity_of_Cell_Shape,Marginal_Adhesion,Single_Epithelial_Cell_Size,Bare_Nuclei,Bland_Chromatin,Normal_Nucleoli,Mitoses
183,0.194620,1.600488,1.609462,1.809684,0.787453,1.768159,1.438379,1.698379,-0.337081
520,-1.239421,-0.695126,-0.748195,-0.642428,-0.106111,-0.690192,-1.006210,-0.621358,-0.337081
200,1.628661,1.272543,1.272654,0.758779,0.787453,1.768159,1.438379,1.698379,0.846400
169,-1.239421,-0.695126,-0.748195,-0.292127,-0.999675,-0.690192,-1.006210,-0.621358,-0.337081
540,0.194620,-0.695126,-0.748195,-0.642428,-0.552893,-0.417042,-0.598779,-0.621358,-0.337081


## Paso 6: Balanceo de clases
Se aplica undersampling solo en train para equilibrar benigno/maligno.

In [7]:
train_processed = X_train_final_df.copy()
train_processed[target_col] = y_train
test_processed = X_test_final_df.copy()
test_processed[target_col] = y_test

print('Distribucion antes del balanceo (train) - conteos:')
print(train_processed[target_col].value_counts().sort_index())
print('Distribucion antes del balanceo (train) - porcentajes:')
print((train_processed[target_col].value_counts(normalize=True).sort_index() * 100).round(2).astype(str) + ' %')

min_count = train_processed[target_col].value_counts().min()
train_balanced = train_processed.groupby(target_col, group_keys=False).sample(n=min_count, random_state=42)

print('Distribucion despues del balanceo (train) - conteos:')
print(train_balanced[target_col].value_counts().sort_index())
print('Distribucion despues del balanceo (train) - porcentajes:')
print((train_balanced[target_col].value_counts(normalize=True).sort_index() * 100).round(2).astype(str) + ' %')

print('Train balanceado:', train_balanced.shape)
print('Test sin balancear:', test_processed.shape)

Distribucion antes del balanceo (train) - conteos:
Class
0    367
1    193
Name: count, dtype: int64
Distribucion antes del balanceo (train) - porcentajes:
Class
0    65.54 %
1    34.46 %
Name: proportion, dtype: str
Distribucion despues del balanceo (train) - conteos:
Class
0    193
1    193
Name: count, dtype: int64
Distribucion despues del balanceo (train) - porcentajes:
Class
0    50.0 %
1    50.0 %
Name: proportion, dtype: str
Train balanceado: (386, 10)
Test sin balancear: (139, 10)


## Paso 7: Conversor a tensores (PyTorch)
En este paso se convierten los conjuntos preprocesados a tensores para usarlos directamente en modelos de PyTorch.

In [8]:
X_train_np = train_balanced.drop(columns=[target_col]).to_numpy().astype(np.float32)
y_train_np = train_balanced[target_col].to_numpy().astype(np.int64)
X_test_np = test_processed.drop(columns=[target_col]).to_numpy().astype(np.float32)
y_test_np = test_processed[target_col].to_numpy().astype(np.int64)

X_train = torch.from_numpy(X_train_np)
y_train_t = torch.from_numpy(y_train_np)
X_test = torch.from_numpy(X_test_np)
y_test_t = torch.from_numpy(y_test_np)

print('Preparacion terminada:')
print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train_t.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_test shape: {y_test_t.shape}')

Preparacion terminada:
X_train shape: torch.Size([386, 9])
y_train shape: torch.Size([386])
X_test shape: torch.Size([139, 9])
y_test shape: torch.Size([139])
